# MDD Challenge 2025 — Training Notebook (Kaggle P100)

**GPU**: P100 (16GB)  
**Run mode**: Set `RUN_STAGE` to control which stage executes.

```
RUN_STAGE = 'stage1'   → Minimal baseline (wav2vec2-vn, no preprocessing)
RUN_STAGE = 'stage2'   → Preprocessing ablations (A–F, one at a time)
RUN_STAGE = 'stage3'   → Model comparison (hubert / wav2vec2-100h)
```

**Score formula**: `Score = 0.5×F1 + 0.4×(1−DER) + 0.1×(1−PER)`  
**Trivial baseline (B0)**: Score = 0.4994 (predict canonical = no errors)

**Fixed split** (never change):  
- Train: 2720 samples, 22 speakers (incl. TUYEN, ADULT)  
- Valid: 460 samples, speakers S0008 (6.2% error) + S0003 (15.4% error)

In [ ]:
# ── Pull code from GitHub ────────────────────────────────────────────────────
import subprocess, os, sys
from pathlib import Path

REPO_URL = 'https://github.com/quangzp/mdd-challenge.git'
BRANCH   = 'feat/giang'
REPO_DIR = Path('/kaggle/working/mdd-challenge')

if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--branch', BRANCH, '--depth', '1',
        REPO_URL, str(REPO_DIR)
    ])
    print(f'Cloned → {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
    print(f'Updated existing clone')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'cwd: {os.getcwd()}')
print('Files:', [p.name for p in sorted(REPO_DIR.iterdir()) if not p.name.startswith('.')])


## 1. Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.40.2', 'accelerate>=0.27'])
print('Install done')

In [ ]:
# ── Stage selector ───────────────────────────────────────────────────────────
RUN_STAGE = 'stage1'   # 'stage1' | 'stage2' | 'stage3'

# Stage 2 ablation selector (only used when RUN_STAGE='stage2')
# Run one at a time; each is independent from Stage 1 baseline
ABLATION = 'C'  # 'A'=normalize_amp | 'B'=trim_silence | 'C'=gain_aug
               # 'D'=noise_aug | 'E'=spec_augment | 'F'=no_adult

# Stage 3 model selector (only used when RUN_STAGE='stage3')
STAGE3_MODEL = 'hubert'  # 'hubert' | 'wav2vec2-100h'

# ── Global hyperparams (identical across all experiments) ───────────────────
EPOCHS       = 10
BATCH_SIZE   = 4
GRAD_ACCUM   = 4         # effective batch = 16
ENCODER_LR   = 2e-5
HEAD_LR      = 2e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_SECONDS  = 15        # truncate audio at 15s
SEED         = 42
BLANK_PENALTIES = [0.0, 0.5, 1.0, 1.5, 2.0]  # sweep at decode time

print(f'Stage: {RUN_STAGE}  |  Ablation: {ABLATION}  |  Model: {STAGE3_MODEL}')

In [ ]:
import os, re, csv, json, math, wave, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoFeatureExtractor, Wav2Vec2ForCTC, HubertForCTC,
    TrainingArguments, Trainer,
    get_linear_schedule_with_warmup, set_seed,
)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Paths & Data Loading

In [ ]:
# ── Resolve paths (local or Kaggle) ─────────────────────────────────────────
def _find(candidates):
    for p in candidates:
        if Path(p).exists(): return Path(p)
    raise FileNotFoundError(f'None found: {candidates}')

DATA_ROOT = _find([
    'MDD-Challenge-2025-training-set',
    '/kaggle/input/datasets/cquangnguynl/mdd-challenge/MDD-Challenge-2025-training-set',
    '/kaggle/input/mdd-challenge/MDD-Challenge-2025-training-set',
])
AUDIO_DIR = DATA_ROOT / 'audio_data' / 'train'
META_DIR  = DATA_ROOT / 'metadata'

# Split files: prefer pre-computed (from stage0_prepare.py),
# otherwise recompute inline
SPLITS_DIR = Path('splits')
SPLITS_DIR.mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'AUDIO_DIR exists: {AUDIO_DIR.exists()}')

In [ ]:
# ── Utilities (inline copy of utils.py for Kaggle self-containedness) ────────
VALID_SPEAKERS = frozenset(['S0008', 'S0003'])
BLANK, UNK = '<blank>', '<unk>'

def get_speaker_id(path):
    stem = Path(path).stem
    m = re.search(r'(S\d+)', stem)
    if m: return m.group(1)
    return 'TUYEN' if 'tuyen' in stem.lower() else 'ADULT'

def norm_phones(s):
    return ' '.join(str(s).replace('*','').replace('$','').split())

def tokenize(s):
    return norm_phones(s).split()

def load_wav_f32(path):
    with wave.open(str(path), 'rb') as wf:
        sr, sw, nch = wf.getframerate(), wf.getsampwidth(), wf.getnchannels()
        frames = wf.readframes(wf.getnframes())
    dtype = {1: np.uint8, 2: np.int16, 4: np.int32}[sw]
    scale = {1: 128.0,   2: 32768.0,  4: 2147483648.0}[sw]
    y = np.frombuffer(frames, dtype=dtype).astype(np.float32)
    y = (y - 128.0)/128.0 if sw==1 else y/scale
    if nch > 1: y = y.reshape(-1, nch).mean(axis=1)
    return y.copy(), sr

def evaluate_on_valid(gt_c, gt_t, preds, tag=''):
    import sys; sys.path.insert(0, str(Path('.').resolve()))
    import evaluate as ev
    gt_p, res_p = '/tmp/_gt.csv', '/tmp/_res.csv'
    with open(gt_p, 'w', newline='') as f:
        w = csv.DictWriter(f, ['canonical','transcript']); w.writeheader()
        for c,t in zip(gt_c, gt_t): w.writerow({'canonical':c,'transcript':t})
    with open(res_p, 'w', newline='') as f:
        w = csv.DictWriter(f, ['predict']); w.writeheader()
        for p in preds: w.writerow({'predict':p})
    f1  = ev.compute_f1( gt_p, res_p)
    per = ev.compute_per(gt_p, res_p)
    der = ev.compute_der(gt_p, res_p)
    sc  = 0.5*f1 + 0.4*(1-der) + 0.1*(1-per)
    if tag: print(f'{tag:45s}  F1={f1:.4f}  PER={per:.4f}  DER={der:.4f}  Score={sc:.4f}')
    return {'f1':f1,'per':per,'der':der,'score':sc}

def greedy_ctc(logits, id2phone, phone2id, penalty=0.0):
    adj = logits.copy()
    adj[..., phone2id[BLANK]] -= penalty
    ids = np.argmax(adj, axis=-1)
    res = []
    for seq in ids:
        out, prev = [], None
        for i in seq:
            if i==prev: continue
            prev=int(i)
            if prev==phone2id[BLANK]: continue
            out.append(id2phone[prev] if prev<len(id2phone) else UNK)
        res.append(' '.join(out))
    return res

def save_results(d, path='results/results_log.json'):
    existing = json.load(open(path)) if Path(path).exists() else {}
    existing.update(d)
    with open(path,'w') as f: json.dump(existing, f, indent=2, ensure_ascii=False)
    print(f'Saved → {path}')

print('Utilities loaded.')

In [ ]:
# ── Stage 0: build split + vocab (runs in <10s) ──────────────────────────────
if not (SPLITS_DIR / 'phone_vocab.json').exists():
    print('Running Stage 0 inline...')
    df_t = pd.read_csv(META_DIR / 'train.csv')
    df_p = pd.read_csv(META_DIR / 'train_phones.csv')
    df_t['speaker_id'] = df_t['path'].map(get_speaker_id)
    df_p['speaker_id'] = df_t['speaker_id'].values
    df_p['c_norm']     = df_p['canonical'].map(norm_phones)
    df_p['t_norm']     = df_p['transcript'].map(norm_phones)
    df_p['ph_error']   = (df_p['c_norm'] != df_p['t_norm'])
    vm = df_t['speaker_id'].isin(VALID_SPEAKERS)
    train_df = df_t[~vm].copy().reset_index(drop=True)
    valid_df = df_t[ vm].copy().reset_index(drop=True)
    train_ph = df_p[~vm].copy().reset_index(drop=True)
    valid_ph = df_p[ vm].copy().reset_index(drop=True)
    # vocab
    counter = Counter()
    for s in train_ph['t_norm']: counter.update(tokenize(s))
    for s in df_p['c_norm']:     counter.update(tokenize(s))
    id2phone = [BLANK, UNK] + sorted(counter.keys())
    phone2id = {p: i for i, p in enumerate(id2phone)}
    # save
    train_df.to_csv(SPLITS_DIR/'train_ids.csv',    index=False)
    valid_df.to_csv(SPLITS_DIR/'valid_ids.csv',    index=False)
    train_ph.to_csv(SPLITS_DIR/'train_phones.csv', index=False)
    valid_ph.to_csv(SPLITS_DIR/'valid_phones.csv', index=False)
    with open(SPLITS_DIR/'phone_vocab.json','w') as f:
        json.dump({'id2phone':id2phone,'phone2id':phone2id}, f, ensure_ascii=False)
    print(f'Stage 0 done. Vocab={len(id2phone)}, Train={len(train_df)}, Valid={len(valid_df)}')
else:
    print('Stage 0 artifacts found, loading...')
    train_df = pd.read_csv(SPLITS_DIR/'train_ids.csv')
    valid_df = pd.read_csv(SPLITS_DIR/'valid_ids.csv')
    train_ph = pd.read_csv(SPLITS_DIR/'train_phones.csv')
    valid_ph = pd.read_csv(SPLITS_DIR/'valid_phones.csv')
    with open(SPLITS_DIR/'phone_vocab.json') as f:
        vd = json.load(f)
    id2phone, phone2id = vd['id2phone'], vd['phone2id']
    print(f'Loaded. Vocab={len(id2phone)}, Train={len(train_df)}, Valid={len(valid_df)}')

# Shared valid ground-truth lists (immutable)
VALID_C = valid_ph['c_norm'].tolist()
VALID_T = valid_ph['t_norm'].tolist()

# Verify no leakage
assert set(train_df['speaker_id']).isdisjoint(VALID_SPEAKERS), 'Speaker leak!'
print('Split assertions pass.')

## 3. Preprocessing Functions

In [ ]:
MAX_SAMPLES = MAX_SECONDS * 16000

# ── Preprocessing ops ────────────────────────────────────────────────────────
def normalize_amp(y, target_peak=0.9):
    peak = np.abs(y).max()
    return y if peak < 1e-6 else y * (target_peak / peak)

def trim_silence(y, sr=16000, threshold_db=-45,
                 frame_ms=25, hop_ms=10, min_dur_sec=0.3):
    fl = int(sr*frame_ms/1000); hl = int(sr*hop_ms/1000)
    e  = [np.mean(y[i:i+fl]**2) for i in range(0, max(1,len(y)-fl), hl)]
    db = 10*np.log10(np.array(e)+1e-10)
    act = db > threshold_db
    if not act.any(): return y
    s = max(0, int(np.argmax(act))*hl - fl)
    e = min(len(y), (len(act)-int(np.argmax(act[::-1])))*hl+fl)
    trimmed = y[s:e]
    return y if len(trimmed)/sr < min_dur_sec else trimmed

def aug_gain(y, lo=-6., hi=6.):
    return np.clip(y * 10**(np.random.uniform(lo, hi)/20.), -1., 1.)

def aug_noise(y, snr_lo=20., snr_hi=35., prob=0.3):
    if np.random.rand() > prob: return y
    sp  = np.mean(y**2) + 1e-10
    np_ = sp / 10**(np.random.uniform(snr_lo, snr_hi)/10.)
    return np.clip(y + np.random.normal(0., np_**0.5, y.shape).astype(np.float32), -1., 1.)

# ── Build audio pipeline based on stage/ablation ────────────────────────────
def build_audio_pipeline(stage, ablation=None, no_adult=False):
    """Returns (preprocess_fn, augment_fn, use_adult) for each experiment.
    All Stage 2 experiments are INDEPENDENT from Stage 1 — not stacked.
    """
    # Stage 1: minimal — only truncation
    def preproc_stage1(y): return y[:MAX_SAMPLES]
    def augment_noop(y):   return y

    if stage == 'stage1':
        return preproc_stage1, augment_noop, True  # include adult

    if stage == 'stage2':
        # Each ablation adds EXACTLY ONE thing over Stage 1 baseline
        if ablation == 'A':  # normalize_amp only
            def p(y): return normalize_amp(y[:MAX_SAMPLES])
            return p, augment_noop, True
        if ablation == 'B':  # trim_silence only
            def p(y): return trim_silence(y, 16000)[:MAX_SAMPLES]
            return p, augment_noop, True
        if ablation == 'C':  # gain augment only
            def p(y): return y[:MAX_SAMPLES]
            def a(y): return aug_gain(y)
            return p, a, True
        if ablation == 'D':  # noise augment only
            def p(y): return y[:MAX_SAMPLES]
            def a(y): return aug_noise(y)
            return p, a, True
        if ablation == 'E':  # spec_augment (handled in model config)
            return preproc_stage1, augment_noop, True
        if ablation == 'F':  # exclude adult data
            return preproc_stage1, augment_noop, False

    # Stage 3: use best_preproc from Stage 2 (update after Stage 2 done)
    # Default: Stage 1 config until Stage 2 results are in
    if stage == 'stage3':
        return preproc_stage1, augment_noop, True

    raise ValueError(f'Unknown stage/ablation: {stage}/{ablation}')

print('Preprocessing functions defined.')

## 4. Dataset & Collator

In [ ]:
class MDDDataset(Dataset):
    def __init__(self, df, ph_df, phone2id, preproc_fn, augment_fn,
                 is_train=True):
        self.df       = df.reset_index(drop=True)
        self.ph       = ph_df.reset_index(drop=True)
        self.p2i      = phone2id
        self.preproc  = preproc_fn
        self.augment  = augment_fn
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ph  = self.ph.iloc[idx]
        y, sr = load_wav_f32(AUDIO_DIR / Path(row['path']).name)
        y = self.preproc(y)
        if self.is_train:
            y = self.augment(y)
        labels = [self.p2i.get(t, self.p2i[UNK])
                  for t in tokenize(ph['t_norm'])]
        return {'input_values': y.astype(np.float32),
                'labels': labels,
                'id': row['id'],
                'ph_error': bool(ph['ph_error'])}


@dataclass
class MDDCollator:
    fe: object
    sr: int = 16000
    pad_id: int = -100

    def __call__(self, batch):
        out = self.fe([b['input_values'] for b in batch],
                      sampling_rate=self.sr, padding=True,
                      return_attention_mask=True,  # critical for CTC
                      return_tensors='pt')
        labels = [torch.tensor(b['labels'], dtype=torch.long) for b in batch]
        maxL   = max(len(l) for l in labels)
        pad    = torch.full((len(labels), maxL), self.pad_id, dtype=torch.long)
        for i, l in enumerate(labels): pad[i, :len(l)] = l
        out['labels'] = pad
        return out

print('Dataset & Collator defined.')

## 5. Model Factory

In [ ]:
MODEL_CONFIGS = {
    'wav2vec2-vn':   'nguyenvulebinh/wav2vec2-base-vietnamese-250h',
    'hubert':        'facebook/hubert-base-ls960',
    'wav2vec2-100h': 'facebook/wav2vec2-base-100h',
}
MODEL_CLASSES = {
    'wav2vec2-vn':   Wav2Vec2ForCTC,
    'hubert':        HubertForCTC,
    'wav2vec2-100h': Wav2Vec2ForCTC,
}

def build_model(model_key, vocab_size, blank_id, enable_spec_augment=False):
    model_name  = MODEL_CONFIGS[model_key]
    model_class = MODEL_CLASSES[model_key]
    fe = AutoFeatureExtractor.from_pretrained(model_name)
    model = model_class.from_pretrained(
        model_name,
        vocab_size=vocab_size,
        pad_token_id=blank_id,
        ctc_loss_reduction='mean',
        ctc_zero_infinity=True,
        ignore_mismatched_sizes=True,
    )
    model.config.apply_spec_augment = enable_spec_augment
    if enable_spec_augment:
        model.config.mask_time_prob    = 0.05
        model.config.mask_feature_prob = 0.0
    else:
        model.config.mask_time_prob    = 0.0
        model.config.mask_feature_prob = 0.0
    model.freeze_feature_encoder()
    return model, fe

def build_optimizer_scheduler(model, n_train, epochs, batch, accum,
                               enc_lr, head_lr, wd, warmup_ratio):
    head_ids    = {id(p) for p in model.lm_head.parameters()}
    enc_params  = [p for p in model.parameters()
                   if id(p) not in head_ids and p.requires_grad]
    head_params = [p for p in model.lm_head.parameters() if p.requires_grad]
    steps  = math.ceil(n_train / (batch * accum)) * epochs
    warmup = int(steps * warmup_ratio)
    opt = torch.optim.AdamW(
        [{'params': enc_params, 'lr': enc_lr},
         {'params': head_params,'lr': head_lr}],
        weight_decay=wd)
    sch = get_linear_schedule_with_warmup(opt, warmup, steps)
    print(f'Optimizer: enc_lr={enc_lr}, head_lr={head_lr}')
    print(f'Steps={steps}, warmup={warmup}')
    print(f'Enc params: {sum(p.numel() for p in enc_params):,}')
    print(f'Head params: {sum(p.numel() for p in head_params):,}')
    return opt, sch

print('Model factory defined.')

## 6. Training & Evaluation Runner

In [ ]:
def run_experiment(exp_name, model_key, train_df, train_ph, valid_df, valid_ph,
                   preproc_fn, augment_fn,
                   enable_spec_augment=False,
                   output_subdir=None):
    """Full train + eval loop. Returns best metrics dict."""
    print(f'\n{"="*60}')
    print(f'Experiment: {exp_name}  |  Model: {model_key}')
    print(f'Train: {len(train_df)}  |  Valid: {len(valid_df)}')
    print(f'SpecAugment: {enable_spec_augment}')
    print(f'{"="*60}')

    out_dir = Path(f'outputs/{output_subdir or exp_name}')
    out_dir.mkdir(parents=True, exist_ok=True)

    # Build model
    model, fe = build_model(model_key, len(id2phone), phone2id[BLANK],
                            enable_spec_augment=enable_spec_augment)

    # Datasets
    train_ds = MDDDataset(train_df, train_ph, phone2id,
                          preproc_fn, augment_fn, is_train=True)
    valid_ds = MDDDataset(valid_df, valid_ph, phone2id,
                          preproc_fn, lambda y: y, is_train=False)
    collator = MDDCollator(fe=fe)

    # Sanity check first batch
    sample_batch = collator([train_ds[0], train_ds[1]])
    with torch.no_grad():
        init_loss = model(**{k: v for k,v in sample_batch.items()}).loss
    print(f'Initial CTC loss (sanity): {float(init_loss):.4f}')
    assert not torch.isnan(init_loss), 'NaN loss at init — check vocab/model'

    # Optimizer + scheduler
    opt, sch = build_optimizer_scheduler(
        model, len(train_df), EPOCHS, BATCH_SIZE, GRAD_ACCUM,
        ENCODER_LR, HEAD_LR, WEIGHT_DECAY, WARMUP_RATIO
    )

    # Training args
    training_args = TrainingArguments(
        output_dir=str(out_dir / 'checkpoints'),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=torch.cuda.is_available(),
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        group_by_length=True,
        remove_unused_columns=False,
        report_to='none',
        logging_steps=30,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        optimizers=(opt, sch),
    )

    train_result = trainer.train()
    print(f'Training complete. Final train loss: {train_result.training_loss:.4f}')

    # ── Decode & evaluate ────────────────────────────────────────────────────
    pred_out = trainer.predict(valid_ds)
    logits   = pred_out.predictions   # (n_valid, T, vocab)

    # Blank frame ratio (diagnostic)
    raw_ids  = np.argmax(logits, axis=-1)
    blank_ratio = (raw_ids == phone2id[BLANK]).mean()
    print(f'Blank frame ratio: {blank_ratio:.3f}')

    # Sweep blank_penalty
    best_score, best_penalty, best_metrics = -1, 0.0, {}
    for penalty in BLANK_PENALTIES:
        preds  = greedy_ctc(logits, id2phone, phone2id, penalty=penalty)
        empty  = sum(1 for p in preds if not p.strip())
        lengths = [len(p.split()) for p in preds]
        metrics = evaluate_on_valid(VALID_C, VALID_T, preds,
                                    f'  {exp_name} penalty={penalty:.1f}')
        print(f'    empty={empty}/{len(preds)}, '
              f'mean_len={np.mean(lengths):.1f}')
        if metrics['score'] > best_score:
            best_score, best_penalty, best_metrics = \
                metrics['score'], penalty, metrics

    print(f'\nBest: penalty={best_penalty}, Score={best_score:.4f}')

    # Per-speaker breakdown (bias check)
    best_preds = greedy_ctc(logits, id2phone, phone2id, penalty=best_penalty)
    for spk in VALID_SPEAKERS:
        mask = valid_df['speaker_id'] == spk
        idx  = [i for i,m in enumerate(mask) if m]
        m = evaluate_on_valid(
            [VALID_C[i] for i in idx],
            [VALID_T[i] for i in idx],
            [best_preds[i] for i in idx],
            f'  {exp_name} [{spk}]'
        )

    # Save model + results
    trainer.save_model(str(out_dir / 'final_model'))
    fe.save_pretrained(str(out_dir / 'final_model'))

    result_record = {
        **best_metrics,
        'best_blank_penalty': best_penalty,
        'blank_frame_ratio': float(blank_ratio),
        'train_loss': float(train_result.training_loss),
        'model': model_key,
        'n_train': len(train_df),
        'n_valid': len(valid_df),
    }
    save_results({exp_name: result_record})
    return result_record

print('Runner defined.')

## 7. Execute Selected Stage

In [ ]:
# ── STAGE 1: Minimal Baseline ─────────────────────────────────────────────────
if RUN_STAGE == 'stage1':
    preproc, augment, use_adult = build_audio_pipeline('stage1')
    t_df = train_df; t_ph = train_ph

    result_s1 = run_experiment(
        exp_name    = 'stage1_baseline',
        model_key   = 'wav2vec2-vn',
        train_df    = t_df, train_ph = t_ph,
        valid_df    = valid_df, valid_ph = valid_ph,
        preproc_fn  = preproc,
        augment_fn  = augment,
    )

    # ── Stage 1 exit check ───────────────────────────────────────────────────
    s = result_s1['score']
    B0 = 0.4994
    print(f'\nStage 1 Score: {s:.4f}  (B0={B0})')
    if s > B0:
        print('✅ Beat B0 — proceed to Stage 2')
    elif s > 0.45:
        print('⚠️  Below B0 but above 0.45 — check blank ratio, try head_lr=5e-3')
    else:
        print('❌ Below 0.45 — debug: check label encoding, CTC loss, vocab')

In [ ]:
# ── STAGE 2: Preprocessing Ablations ──────────────────────────────────────────
# Each ablation is INDEPENDENT (starts from Stage 1 config, not stacked)
# Run this cell multiple times with different ABLATION values
if RUN_STAGE == 'stage2':
    enable_spec = (ABLATION == 'E')
    preproc, augment, use_adult = build_audio_pipeline('stage2', ablation=ABLATION)

    t_df = train_df[train_df['speaker_id'] != 'ADULT'].reset_index(drop=True) \
           if not use_adult else train_df
    t_ph = train_ph[train_ph['speaker_id'] != 'ADULT'].reset_index(drop=True) \
           if not use_adult else train_ph

    result_s2 = run_experiment(
        exp_name          = f'stage2_ablation_{ABLATION}',
        model_key         = 'wav2vec2-vn',
        train_df          = t_df, train_ph = t_ph,
        valid_df          = valid_df, valid_ph = valid_ph,
        preproc_fn        = preproc,
        augment_fn        = augment,
        enable_spec_augment = enable_spec,
    )

    # Load Stage 1 baseline for comparison
    log = json.load(open('results/results_log.json')) \
          if Path('results/results_log.json').exists() else {}
    s1_score = log.get('stage1_baseline', {}).get('score', None)
    delta = result_s2['score'] - s1_score if s1_score else float('nan')

    print(f'\nAblation {ABLATION}: Score={result_s2["score"]:.4f}')
    if s1_score:
        print(f'Stage 1 baseline: {s1_score:.4f}')
        print(f'Delta: {delta:+.4f}')
        if delta >= 0.010:
            print('✅ Improvement ≥ 0.010 — KEEP this preprocessing step')
        elif delta >= 0:
            print('⚠️  Small improvement < 0.010 — borderline, check bias')
        else:
            print('❌ Negative delta — do NOT include this step')

In [ ]:
# ── STAGE 3: Model Comparison ──────────────────────────────────────────────────
# Run after Stage 2 is complete; update build_audio_pipeline('stage3') with
# the best preprocessing combination found in Stage 2.
#
# To update: edit build_audio_pipeline to apply kept Stage 2 steps in stage3 branch
if RUN_STAGE == 'stage3':
    preproc, augment, use_adult = build_audio_pipeline('stage3')
    t_df = train_df; t_ph = train_ph

    result_s3 = run_experiment(
        exp_name   = f'stage3_{STAGE3_MODEL}',
        model_key  = STAGE3_MODEL,
        train_df   = t_df, train_ph = t_ph,
        valid_df   = valid_df, valid_ph = valid_ph,
        preproc_fn = preproc,
        augment_fn = augment,
    )

    # Print comparison table
    log = json.load(open('results/results_log.json')) \
          if Path('results/results_log.json').exists() else {}
    print('\n=== Stage 3 Model Comparison ===')
    print(f'{"Model":30s}  {"F1":8s}  {"PER":8s}  {"DER":8s}  {"Score":8s}')
    print('-'*65)
    for k in ['stage2_best_wav2vec2_vn', f'stage3_{STAGE3_MODEL}']:
        if k in log:
            r = log[k]
            print(f'{k:30s}  {r["f1"]:.4f}    {r["per"]:.4f}    {r["der"]:.4f}    {r["score"]:.4f}')

## 8. Results Summary

Chạy cell này sau khi tất cả experiments xong để xem bảng tổng hợp.

In [ ]:
if Path('results/results_log.json').exists():
    log = json.load(open('results/results_log.json'))
    B0 = log.get('naive_B0_canonical', {}).get('score', 0.4994)
    SEP = '='*70
    print(SEP)
    print(f"{'Experiment':35s}  {'F1':7s}  {'PER':7s}  {'DER':7s}  {'Score':7s}  {'dB0':7s}")
    print('-'*70)
    print(f"{'B0 canonical (trivial)':35s}  {0.0:.4f}  {0.0056:.4f}  {0.0:.4f}  {B0:.4f}  {0.0:+.4f}")
    skip = {'naive_B0_canonical','naive_B1_oracle','naive_B2_empty','split_info'}
    for k, v in log.items():
        if k in skip or not isinstance(v, dict) or 'score' not in v:
            continue
        delta = v['score'] - B0
        f1  = v.get('f1',  0)
        per = v.get('per', 0)
        der = v.get('der', 0)
        sc  = v['score']
        print(f"{k:35s}  {f1:.4f}  {per:.4f}  {der:.4f}  {sc:.4f}  {delta:+.4f}")
    print(SEP)
    print('Upper bound (oracle): Score=1.0000')
else:
    print('No results yet — run experiments first.')
